# Пайплайн: обучение ранкера и ответы на test_public (и другой тестовый сплит) — Google Colab

Тот же код, что `python -m recsys …` (README). Ячейки — **сверху вниз**. Код берётся из ветки `dev/baseline`
каждую сессию заново; всё, что считается (кэш, ранкер, отчёты, сабмиты), пишется на Google Drive в `MyDrive/recsys/`
и переживает отключение Colab: повторный запуск берёт готовое с диска.

**Перед запуском:** Runtime → Change runtime type → **CPU, High-RAM** (GPU не нужен). На Drive в `MyDrive/recsys/`:
`music4all_crs/` (parquet датасета) и файлы башен для лучшего прогона (`configs/best.yaml`): `artifacts/twotower/v2/`,
`artifacts/twotower/lazy_e0/` (векторы — `twotower.ipynb`, `lazy.ipynb`) и `artifacts/twotower_pavel/` (готовые top-200).

| шаг | ячейки | время в Colab |
|---|---|---|
| 0. подготовка | 1–3 | ~3 мин, в каждой новой сессии |
| 1. обучение ранкера и метрики | 4–5 | первый раз ~1.5 ч, дальше — минуты (кэш) |
| 2. ответы на test_public | 6 | первый раз несколько часов (273k запросов), при обрыве — продолжит |
| 3. ответы на другой сплит | 7 | как шаг 2 |
| выключить машину | 8 | — |

## 0. Подготовка

In [ ]:
# 1. Drive, код (ветка dev/baseline), датасет на диск машины; кэш, прогоны, artifacts — ссылками на Drive
import os, subprocess
from google.colab import drive
drive.mount("/content/drive")
REPO_URL = "https://github.com/rimgro/conversational-recsys.git"
BRANCH = "dev/baseline"
DRIVE_DIR = "/content/drive/MyDrive/recsys"
REPO_DIR = "/content/recsys-baseline"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run(["git", "pull", "--ff-only"], check=True)

def sync_data():
    """Датасет с Drive на диск машины (с Drive чтение медленное); новые файлы (другой сплит) докачивает."""
    subprocess.run(["rsync", "-a", f"{DRIVE_DIR}/music4all_crs/", "music4all_crs/"], check=True)

sync_data()
for d in ("cache", "runs", "artifacts"):
    os.makedirs(f"{DRIVE_DIR}/{d}", exist_ok=True)
    if not os.path.islink(d):
        os.symlink(f"{DRIVE_DIR}/{d}", d)
print("ветка:", subprocess.run(["git", "branch", "--show-current"], capture_output=True, text=True).stdout.strip(),
      "| файлы датасета:", sorted(os.listdir("music4all_crs")))

In [ ]:
# 2. пакеты
%pip install -q -r requirements.txt

In [ ]:
# 3. параметры прогона: все команды ниже берут их отсюда
RUN = "best"           # runs/<RUN>/ на Drive: ранкер, отчёты, сабмиты. Новый эксперимент — новое имя
SOURCES = ""           # кандгены поверх configs/default.yaml: "" — как есть; "+twotower"; "+twotower,-tfidf"
CONFIG = "configs/best.yaml"   # лучший прогон: 11 кандгенов team1 + три башни; "" — configs/default.yaml как есть
THREADS = os.cpu_count()

OPTS = f"--run {RUN} --set ranker.threads={THREADS}"
OPTS += f" --sources {SOURCES}" if SOURCES else ""
OPTS += f" --config {CONFIG}" if CONFIG else ""
print("опции:", OPTS)

## 1. Обучение ранкера и метрики
Кандгены → отбор 150 → признаки → CatBoost на `ranker_train`, early stopping на `valid`; отчёт на `valid` и `test`
(сэмпл test_public, 10k). Остановилось — запустить ячейку снова: индексы, кандгены и признаки возьмутся с диска.

Включили новый кандген (`SOURCES`) — сначала подобрать ему вес в отборе, иначе он будет только признаком:
```
!python -m recsys tune-policy {OPTS}        # -> runs/<RUN>/selection.yaml; затем CONFIG = f"runs/{RUN}/selection.yaml"
```

In [ ]:
# 4. обучение и отчёт
!python -m recsys run {OPTS}

In [ ]:
# 5. отчёт на test: каждый кандген, отбор, ранкер, по типам запросов
from IPython.display import Markdown
Markdown(open(f"runs/{RUN}/test/report.md", encoding="utf8").read())

## 2. Ответы на test_public
Все запросы test_public -> `runs/<RUN>/submission_test_public.parquet` (query_id, top20, response). Тот же прогон
(`RUN`, `SOURCES`, `CONFIG`), что при обучении, — иначе ранкер увидит другие признаки.

In [ ]:
# 6. сабмит test_public и проверка формата
import pandas as pd

def check(path):
    sub = pd.read_parquet(path)
    ok = sub["query_id"].is_unique and sub["top20"].map(lambda x: len(x) == 20 and len(set(x)) == 20).all()
    print(path, "| строк:", len(sub), "| формат ok:" if ok else "| ФОРМАТ НЕ ТОТ:", ok,
          "| пустых ответов:", int((sub["response"].str.len() == 0).sum()))
    return sub

!python -m recsys submit {OPTS}
check(f"runs/{RUN}/submission_test_public.parquet").head()

## 3. Ответы на другой тестовый сплит (например, test_private)
Положить на Drive в `MyDrive/recsys/music4all_crs/` файлы `<сплит>.parquet` (пользователи: история, профиль) и
`<сплит>_queries.parquet` (запросы), ответы не нужны. Граница истории — `data.windows.<сплит>` в конфиге (test_private
уже есть). Ранкер и индексы — те же, что для test_public. С башнями: сначала векторы этих запросов на GPU —
`twotower.ipynb` / `lazy.ipynb`, `encode_twotower.py --parts submit --set data.parts.test_split=<сплит>`; списки
`twotower_pavel` для сплита считаются вне репозитория (без них прогон остановится).

In [ ]:
# 7. сабмит другого сплита
SPLIT = "test_private"
sync_data()                                    # докачать новые файлы с Drive
assert os.path.exists(f"music4all_crs/{SPLIT}_queries.parquet"), f"нет music4all_crs/{SPLIT}_queries.parquet на Drive"
!python -m recsys submit {OPTS} --test-split {SPLIT}
check(f"runs/{RUN}/submission_{SPLIT}.parquet").head()

## Выключить машину (Drive сохраняется)

In [ ]:
# 8.
from google.colab import runtime
drive.flush_and_unmount()
runtime.unassign()